In [ ]:
import json
import matplotlib.pyplot as plt

In [ ]:
#load json file
with open("/Users/jost/Jost/Code/2024-nc-hackathon-spades/spades_label/points.json", 'r') as f:
    data = json.load(f)
    keypoints = data['keypoints']
    mesh_faces = data['mesh_faces']
    print(mesh_faces)

In [ ]:
# creat a 3d diagram of the keypoints
import matplotlib.pyplot as plt
from mpl_toolkits.mplot3d import Axes3D
from mpl_toolkits.mplot3d.art3d import Poly3DCollection
import random


fig = plt.figure()
ax = fig.add_subplot(111, projection='3d')
# plot the keypoints
for [x,y,z] in keypoints:
    ax.scatter(x, y, z, c='r', marker='o')
# plot the mesh faces
faces = [[keypoints[i] for i in face] for face in mesh_faces]
colors = [f"#{random.randint(0, 0xFFFFFF):06x}" for _ in faces]
mesh = Poly3DCollection(faces, alpha=0.5, facecolors=colors)
ax.add_collection3d(mesh)

# Draw edges for each face
for face in faces:
    for i in range(len(face)):
        start = face[i]
        end = face[(i + 1) % len(face)]  # Connect to the next vertex, looping back to the first
        ax.plot([start[0], end[0]], [start[1], end[1]], [start[2], end[2]], color='k', linestyle='--')


ax.set_box_aspect([1,1,1])  # Set equal aspect ratio
# show the plot
plt.show()


In [ ]:
import numpy as np
import cv2
from scipy.spatial.transform import Rotation as R

pos = [0,0,0]
quat = [1,0,0,0]

with open("/Users/jost/Jost/Code/2024-nc-hackathon-spades/spades_label/camera.json", 'r') as f:
    data = json.load(f)
    K_mat = np.array(data['cameraMatrix'])

R_mat = R.from_quat(quat).as_matrix()
T_mat = np.array(pos).reshape(3,1)

X_cam = np.array(keypoints) @ R_mat.T + T_mat.T

uv_h = (K_mat @ X_cam.T).T

uv = uv_h[:, :2] / uv_h[:, 2:3]

print(uv)

In [ ]:
# plot an array of 8 points over an image
def plot_points(points, img_path, ax = None):
    if ax is None:
        fig, ax = plt.subplots()
    img = cv2.imread(img_path)
    img = cv2.cvtColor(img, cv2.COLOR_BGR2RGB)
    ax.imshow(img)
    for (x,y) in points:
        ax.scatter(x, y, c='r', marker='o')
    return ax

In [ ]:
label_path = "/Users/jost/Downloads/synthetic/synthetic/labels/RT000.csv"
no = 0

import pandas as pd

labels = pd.read_csv(label_path).to_records(index=False)
(name, Tx, Ty, Tz, Qx, Qy, Qz, Qw) = labels[no]
print(name, Tx, Ty, Tz, Qx, Qy, Qz, Qw)

In [ ]:
with open("/Users/jost/Jost/Code/2024-nc-hackathon-spades/spades_label/camera.json", 'r') as f:
    data = json.load(f)
    K_mat = np.array(data['cameraMatrix'])
quat = [Qx, Qy, Qz, Qw]
pos = [Tx, Ty, Tz]

R_cam = R.from_quat(quat).as_matrix()
t_cam = np.array(pos).reshape(3,1)

pts_obj = np.asarray(keypoints, dtype=np.float64)*2.5

X_cam = pts_obj @ R_cam.T + t_cam.T

print("min / max Z_cam:", X_cam[:, 2].min(), X_cam[:, 2].max())
assert np.all(X_cam[:, 2] > 0), "Some points are behind the camera!"

uv_h = (K_mat @ X_cam.T).T                                   # N×3  (u',v',w')
uv   = uv_h[:, :2] / uv_h[:, 2:3]

print("pixel coords:\n", uv)

In [ ]:
img_path = "/Users/jost/Jost/Code/2024-nc-hackathon-spades/generated_dataset/lnes/RT000/img000_RT000.png"
ax = plot_points(uv, img_path)
ax.set_title(name)
plt.show()

In [ ]:
# first, find the minimal square that contains all points
def find_bounding_square(points):
    points = np.array(points)
    min_x, min_y = points.min(axis=0)
    max_x, max_y = points.max(axis=0)

    center_x = (min_x + max_x) / 2
    center_y = (min_y + max_y) / 2

    side_length = max(max_x - min_x, max_y - min_y)

    half_side = side_length / 2

    square = [
        (center_x - half_side, center_y - half_side),
        (center_x + half_side, center_y - half_side),
        (center_x + half_side, center_y + half_side),
        (center_x - half_side, center_y + half_side)
    ]

    return square

# crop the image to the bounding square
img = cv2.imread(img_path)
img = cv2.cvtColor(img, cv2.COLOR_BGR2RGB)
square = find_bounding_square(uv)
# crop the square from the image
def crop_square(img, square):
    min_x = int(min(p[0] for p in square))
    max_x = int(max(p[0] for p in square))
    min_y = int(min(p[1] for p in square))
    max_y = int(max(p[1] for p in square))

    #get center of this square
    cx, cy = (max_x + min_x)/2, (max_y + min_y)/2

    return (cx,cy), img[min_y:max_y, min_x:max_x]

center, cropped_img = crop_square(img, square)

# plot both imgs
fig, axs = plt.subplots(1, 2, figsize=(12, 6))
axs[0].imshow(img)
axs[0].set_title("Original Image")
axs[0].scatter(center[0], center[1], c='r', marker='o', s=50)
axs[0].scatter(*zip(*uv), c='b', marker='o', s=10)
axs[1].imshow(cropped_img)
axs[1].set_title("Cropped Image")
plt.show()

In [ ]:
# now scale the cropped image to 224x224 and return the factor
def scale_image_to_square(img, target_size=224):
    h, w, _ = img.shape
    scale_factor = target_size / max(h, w)
    new_h, new_w = int(h * scale_factor), int(w * scale_factor)
    scaled_img = cv2.resize(img, (new_w, new_h))

    # Create a square canvas
    square_img = np.zeros((target_size, target_size, 3), dtype=np.uint8)

    # Calculate the position to place the scaled image in the center
    start_x = (target_size - new_w) // 2
    start_y = (target_size - new_h) // 2

    # Place the scaled image on the canvas
    square_img[start_y:start_y + new_h, start_x:start_x + new_w] = scaled_img

    return square_img, scale_factor

scaled_image, scale_factor = scale_image_to_square(cropped_img)
# Plot the scaled image
plt.figure(figsize=(6, 6))
plt.imshow(scaled_image)
plt.title("Scaled Image")
plt.show()

In [ ]:
scale_factor